# Caso Práctico: Preprocesamiento de Series temporales

En este caso práctico veremos cómo aplicar diferentes técnicas de preprocesamiento para series temporales con el fin de que los datos temporales presenten propiedades más apropiadas para los modelos de ML.

El contenido de este cuaderno está estructurado en:

1- Importación de librerías
2- Lectura de datos
3- Escalado
4- Transformación de datos
5- Feature engineering

# Importación de librerías

In [1]:
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy.stats import boxcox
from scipy.stats import yeojohnson
from scipy.stats import normaltest

# Lectura de datos

In [2]:
csv_path = 'AirPassengers.csv'
df = pd.read_csv(csv_path)
df.head()

,Month,#Passengers
0,1949-01,112
1,1949-02,118
2,1949-03,132
3,1949-04,129
4,1949-05,121


In [3]:
df['Fecha'] = df['Month']
df['NumerPasajeros'] = df['#Passengers']

fig = px.line(df, x=df['Fecha'], y=df['NumerPasajeros'],
              markers=True,
              labels="fdsf",
              title="Número Pasajeros")
fig.show()

# Escalado

In [4]:
minmax_scaler = MinMaxScaler()
starndard_scaler = StandardScaler()

df['Min-Max-NumeroPasajeros'] = minmax_scaler.fit_transform(df['NumerPasajeros'].values.reshape(-1,1))
df['Estandar-NumeroPasajeros'] = starndard_scaler.fit_transform(df['NumerPasajeros'].values.reshape(-1,1))

fig = make_subplots(rows=2, cols=1,
                    shared_xaxes=True,
                    vertical_spacing=0.08,
                    )

fig.add_trace(go.Scatter(y=df['Min-Max-NumeroPasajeros'], name="Escalado Min-Max", mode="markers+lines"),
              row=1, col=1)

fig.add_trace(go.Scatter(y=df['Estandar-NumeroPasajeros'], name="Estandarización", mode="markers+lines"),
              row=2, col=1)

fig.update_layout(height=500, width=900,
                  title_text="Ejemplos de escalado"
                  )

fig['layout']['xaxis2']['title']='Fecha'
fig['layout']['yaxis1']['title']='Nº Pasajeros'
fig['layout']['yaxis2']['title']='Nº Pasajeros'
fig.show()

In [5]:
fig = px.histogram(df['NumerPasajeros'],
                   nbins=50,
                   title="Número de pasajeros")
fig.show()

In [6]:
from scipy.stats import anderson
from scipy.stats import kstest

p_original = anderson(df['NumerPasajeros'])
result_ks = kstest(df['NumerPasajeros'], 'norm')

print(p_original)
print(result_ks)

AndersonResult(statistic=1.818533152167305, critical_values=array([0.561, 0.639, 0.767, 0.894, 1.064]), significance_level=array([15. , 10. ,  5. ,  2.5,  1. ]), fit_result=  params: FitParams(loc=280.2986111111111, scale=119.96631694294321)
 success: True
 message: '`anderson` successfully fit the distribution to the data.')
KstestResult(statistic=1.0, pvalue=0.0, statistic_location=104, statistic_sign=-1)


# Transformaciones

In [7]:
df['LogNatural-NumeroPasajeros'] = np.log(df['NumerPasajeros'].values)
df['Log10-NumeroPasajeros'] = np.log10(df['NumerPasajeros'].values)

In [8]:
minmax_scaler = MinMaxScaler()
starndard_scaler = StandardScaler()

df['Min-Max-NumeroPasajeros'] = minmax_scaler.fit_transform(df['NumerPasajeros'].values.reshape(-1,1))
df['Estandar-NumeroPasajeros'] = starndard_scaler.fit_transform(df['NumerPasajeros'].values.reshape(-1,1))

fig = make_subplots(rows=1, cols=1,
                    shared_xaxes=True,
                    vertical_spacing=0.08,
                    )

fig.add_trace(go.Scatter(y=df['LogNatural-NumeroPasajeros'], name="Logaritmo Natural", mode="markers+lines"),
              row=1, col=1)

fig.add_trace(go.Scatter(y=df['Log10-NumeroPasajeros'], name="Logaritmo base 10", mode="markers+lines"),
              row=1, col=1)

fig.update_layout(height=500, width=900,
                  title_text="Transformaciones logarítmicas"
                  )

fig['layout']['xaxis1']['title']='Fecha'
fig['layout']['yaxis1']['title']='Nº Pasajeros'

fig.show()

In [9]:
fig = px.histogram(df, x=['LogNatural-NumeroPasajeros', 'Log10-NumeroPasajeros'], nbins=50)
fig.show()

In [10]:
_, p_10 = normaltest(df['Log10-NumeroPasajeros'])
_, p_nat = normaltest(df['LogNatural-NumeroPasajeros'])
print(f"Significance:\n    - Log 10: {p_10:.5f}\n    - Log Natural: {p_nat:.5f}")

Significance:
    - Log 10: 0.00004
    - Log Natural: 0.00004


In [11]:
p_10 = anderson(df['Log10-NumeroPasajeros'])
p_nat = anderson(df['LogNatural-NumeroPasajeros'])
result_ks_10 = kstest(df['Log10-NumeroPasajeros'], 'norm')
result_ks_nat = kstest(df['LogNatural-NumeroPasajeros'], 'norm')

print(f"-----  Log base 10  -----\n    - Anderson:\n {p_10}\n    - Kolmogorov:\n {result_ks_10}")
print(f"\n\n\n-----  Log Natural  -----\n    - Anderson:\n {p_nat}\n    - Kolmogorov:\n {result_ks_nat}")


-----  Log base 10  -----
    - Anderson:
 AndersonResult(statistic=1.066789046997485, critical_values=array([0.561, 0.639, 0.767, 0.894, 1.064]), significance_level=array([15. , 10. ,  5. ,  2.5,  1. ]), fit_result=  params: FitParams(loc=2.4069364365272574, scale=0.19172208903928986)
 success: True
 message: '`anderson` successfully fit the distribution to the data.')
    - Kolmogorov:
 KstestResult(statistic=0.9781539828821812, pvalue=1.4812788449271694e-239, statistic_location=2.0170333392987803, statistic_sign=-1)



-----  Log Natural  -----
    - Anderson:
 AndersonResult(statistic=1.066789046997485, critical_values=array([0.561, 0.639, 0.767, 0.894, 1.064]), significance_level=array([15. , 10. ,  5. ,  2.5,  1. ]), fit_result=  params: FitParams(loc=5.542175958531871, scale=0.441456424219546)
 success: True
 message: '`anderson` successfully fit the distribution to the data.')
    - Kolmogorov:
 KstestResult(statistic=0.9999982945938429, pvalue=0.0, statistic_location=4.6443908

In [12]:
df['BoxCox-NumeroPasajeros'], boxcox_lambda = boxcox(df['NumerPasajeros'].values)
df['YeoJohnson-NumeroPasajeros'], yeojohnson_lambda = yeojohnson(df['NumerPasajeros'].values)

In [13]:
fig = make_subplots(rows=1, cols=1,
                    shared_xaxes=True,
                    # shared_yaxes=True,
                    vertical_spacing=0.08,
                    )

fig.add_trace(go.Scatter(y=df['BoxCox-NumeroPasajeros'], name=f"Box-Cox (lambda = {boxcox_lambda:.4f})", mode="markers+lines"),
              row=1, col=1)

fig.add_trace(go.Scatter(y=df['YeoJohnson-NumeroPasajeros'], name=f"Yeo-Johnson (lambda = {yeojohnson_lambda:.4f})", mode="markers+lines"),
              row=1, col=1)

fig.update_layout(height=500, width=900,
                  title_text="Transformaciones en Potencia"
                  )

# fig['layout']['xaxis1']['title']='Retardo (Intervalos)'
fig['layout']['xaxis1']['title']='Fecha'
fig['layout']['yaxis1']['title']='Nº Pasajeros'
# fig['layout']['yaxis2']['title']='Nº Pasajeros'
fig.show()

In [14]:
fig = px.histogram(df, x=['BoxCox-NumeroPasajeros', 'YeoJohnson-NumeroPasajeros'], nbins=10)
fig.show()

In [15]:
_, p_boxcox = normaltest(df['BoxCox-NumeroPasajeros'])
_, p_yeojohnson = normaltest(df['YeoJohnson-NumeroPasajeros'])
print(f"Significance:\n    - Box-Cox: {p_boxcox:.5f}\n    - Yeo-Johnson: {p_yeojohnson:.5f}")

Significance:
    - Box-Cox: 0.00005
    - Yeo-Johnson: 0.00005


In [16]:
p_boxcox = anderson(df['BoxCox-NumeroPasajeros'])
p_yeojohnson = anderson(df['YeoJohnson-NumeroPasajeros'])
result_ks_boxcox = kstest(df['BoxCox-NumeroPasajeros'], 'norm')
result_ks_yeojohnson = kstest(df['YeoJohnson-NumeroPasajeros'], 'norm')

print(f"-----  Box-Cox  -----\n    - Anderson:\n {p_boxcox}\n    - Kolmogorov:\n {result_ks_boxcox}")
print(f"\n\n\n-----  Yeo-Johnson  -----\n    - Anderson:\n {p_yeojohnson}\n    - Kolmogorov:\n {result_ks_yeojohnson}")



-----  Box-Cox  -----
    - Anderson:
 AndersonResult(statistic=0.9714498405606378, critical_values=array([0.561, 0.639, 0.767, 0.894, 1.064]), significance_level=array([15. , 10. ,  5. ,  2.5,  1. ]), fit_result=  params: FitParams(loc=8.621254402002657, scale=1.0007255519708933)
 success: True
 message: '`anderson` successfully fit the distribution to the data.')
    - Kolmogorov:
 KstestResult(statistic=0.9999999999879957, pvalue=0.0, statistic_location=6.679300529915853, statistic_sign=-1)



-----  Yeo-Johnson  -----
    - Anderson:
 AndersonResult(statistic=0.9725953776470533, critical_values=array([0.561, 0.639, 0.767, 0.894, 1.064]), significance_level=array([15. , 10. ,  5. ,  2.5,  1. ]), fit_result=  params: FitParams(loc=8.55540518125662, scale=0.9820313930919445)
 success: True
 message: '`anderson` successfully fit the distribution to the data.')
    - Kolmogorov:
 KstestResult(statistic=0.9999999999854101, pvalue=0.0, statistic_location=6.650651976103616, statistic_sign=

In [17]:
p_yeojohnson

AndersonResult(statistic=0.9725953776470533, critical_values=array([0.561, 0.639, 0.767, 0.894, 1.064]), significance_level=array([15. , 10. ,  5. ,  2.5,  1. ]), fit_result=  params: FitParams(loc=8.55540518125662, scale=0.9820313930919445)
 success: True
 message: '`anderson` successfully fit the distribution to the data.')

In [18]:
p_boxcox

AndersonResult(statistic=0.9714498405606378, critical_values=array([0.561, 0.639, 0.767, 0.894, 1.064]), significance_level=array([15. , 10. ,  5. ,  2.5,  1. ]), fit_result=  params: FitParams(loc=8.621254402002657, scale=1.0007255519708933)
 success: True
 message: '`anderson` successfully fit the distribution to the data.')

# Feature Engineering

In [19]:
def create_time_delay_embedding(series, delay=1, embedding_dimension=2):
    n = len(series)
    embedded_series = np.zeros((n - (embedding_dimension - 1) * delay, embedding_dimension))
    
    for i in range(embedding_dimension):
        embedded_series[:, i] = series[i * delay:i * delay + len(embedded_series)]
    
    return embedded_series


def create_input_target(series, delay=1, n_lag=1):
    embedding_dimension = n_lag + 1
    delay_embedding = create_time_delay_embedding(series,
                                                  delay=delay,
                                                  embedding_dimension=embedding_dimension)
    X = delay_embedding[:, :-1]
    y = delay_embedding[:, -1]
    return X, y

np.random.seed(0)
time_series = np.random.randn(100)

# Create input features and target values
X, y = create_input_target(time_series, delay=1, n_lag=3)

print("Input features shape (X):", X.shape)
print("Target values shape (y):", y.shape)


Input features shape (X): (97, 3)
Target values shape (y): (97,)


Los datos fueron preparados antes de utilizar cualquier modelo. Se abrió el archivo con los datos de los pasajeros del avión, se revisaron los valores para detectar si había algún error o dato extraño, se aplicó una transformación logarítmica para hacer los datos más estables y fáciles de trabajar, y también se escalaron los valores para que todos quedaran en un rango similar (de 0 a 1). Después de eso, se crearon nuevas columnas utilizando los valores anteriores de la serie (lo que se conoce como time-delay embedding), lo cual ayuda al modelo a aprender a partir de lo que ocurrió en el pasado.